In [1]:
!pip install -qU langchain-community pypdf
!pip install -qU langchain langchain-huggingface sentence_transformers


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from pprint import pprint

In [3]:
file_path="/Attention Is All You Need - 2017 (1706.03762).pdf"
loader=PyPDFLoader(file_path)
doc=loader.load()
pprint(doc[0].metadata)
pprint(doc[0].page_content)



{'author': '',
 'creationdate': '2023-08-03T00:07:29+00:00',
 'creator': 'LaTeX with hyperref',
 'keywords': '',
 'moddate': '2023-08-03T00:07:29+00:00',
 'page': 0,
 'page_label': '1',
 'producer': 'pdfTeX-1.40.25',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live '
                    '2023) kpathsea version 6.3.5',
 'source': '/Attention Is All You Need - 2017 (1706.03762).pdf',
 'subject': '',
 'title': '',
 'total_pages': 15,
 'trapped': '/False'}
('Provided proper attribution is provided, Google hereby grants permission to\n'
 'reproduce the tables and figures in this paper solely for use in '
 'journalistic or\n'
 'scholarly works.\n'
 'Attention Is All You Need\n'
 'Ashish Vaswani∗\n'
 'Google Brain\n'
 'avaswani@google.com\n'
 'Noam Shazeer∗\n'
 'Google Brain\n'
 'noam@google.com\n'
 'Niki Parmar∗\n'
 'Google Research\n'
 'nikip@google.com\n'
 'Jakob Uszkoreit∗\n'
 'Google Research\n'
 'usz@google.com\n'
 'Llion Jones∗\n'
 'Google Research\n'
 'll

In [4]:
!pip install -qU langchain-text-splitters


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
text_config=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
all_splits=text_config.split_documents(doc)
pprint(len(all_splits))

52


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

In [8]:
embedding_model=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
!pip install -qU langchain-chroma

In [10]:
from langchain_chroma import Chroma
vector_store=Chroma(
    collection_name="attention_google_doc",
    embedding_function=embedding_model,
    persist_directory="./chroma_langchain_db"

)
document_ids=vector_store.add_documents(documents=all_splits)


In [22]:
def retrieve_context(query: str,k: int=2):
  retrieved_docs=vector_store.similarity_search(query,k=k)

  docs_content=""
  for i in retrieved_docs:
    docs_content+=f"source:{i.metadata}\n"
    docs_content+=f"Content:{i.page_content}\n\n"

  return  docs_content,retrieved_docs

In [12]:
!pip install -U  langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.3 MB/s eta 0:00:00


In [18]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

In [19]:
api_key=userdata.get("GEMINI_API_KEY")
model=init_chat_model(
    "google_genai:gemini-2.5-flash",
    api_key=api_key,
)


In [23]:
def docu_chat(user_query):
  context,source_docs=retrieve_context(user_query,k=2)
  system_message=f"""You are a helpful chatbot.
                     Use only the following pieces of context to answer the
                     question. Don't makeup any new information: {context} """
  message=[
      {"role":"system","content":system_message},
      {"role":"user","content":user_query},
  ]
  response=model.invoke(message)
  return {
      "answer":response.content,
      "source_data":source_docs,
      "content_used":context
  }

In [26]:
print(docu_chat("explain what is the use of decoders in transformer")["answer"])

The Transformer model architecture includes a decoder, which is shown in the right half of Figure 1. It uses stacked self-attention and point-wise, fully connected layers.

Specifically, the decoder's use involves:
*   **Encoder-decoder attention layers:** In these layers, queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence.
*   **Self-attention layers:** These layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position.
